# 13 · Synthetic Clinical Document Generation

**Purpose:** Generate synthetic breast-imaging + pathology source documents and convert them to realistic scanned PDFs for testing the OCR preprocessing pipeline.

Adapted from the Microsoft *LLM Data Creation* framework (EMNLP'23).

### Pipeline
```
Claude LLM
  └─► synthetic OCR-style .txt  ──► reportlab clean PDF
                                         └─► fitz render pages → numpy images
                                                  └─► cv2 scan degradation (good/medium/poor)
                                                           └─► scanned PDF + ground-truth JSON
```

### Outputs
| Path | Description |
|------|-------------|
| `data/synthetic/{setting}/{id}_document.txt` | Raw synthetic clinical text |
| `data/synthetic/{setting}/{id}_features.json` | Ground-truth feature labels |
| `data/synthetic/pdfs/{id}_{quality}.pdf` | Scanned-look PDF at good/medium/poor quality |

In [ ]:
# Install / verify dependencies
import subprocess, sys

def pip(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=True)

try:
    import reportlab
except ImportError:
    pip("reportlab")

try:
    import anthropic
except ImportError:
    pip("anthropic")

print("All dependencies available.")

In [ ]:
import json, os, random, re, textwrap, uuid, warnings
from pathlib import Path

import anthropic
import cv2
import fitz                          # PyMuPDF
import matplotlib.pyplot as plt
import numpy as np
from dotenv import load_dotenv
from PIL import Image
from reportlab.lib.pagesizes import LETTER
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer
from reportlab.lib.enums import TA_LEFT

warnings.filterwarnings("ignore")
load_dotenv(override=True)

print("Imports OK")

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
REPO_ROOT   = Path(r"C:\Users\jamesr4\OneDrive - Memorial Sloan Kettering Cancer Center\Documents\GitHub\llm_summarization_br_ca")
OCR_CACHE   = Path(r"C:\Users\jamesr4\loc\data_private\ocr_cache")   # real seeds
SYN_ROOT    = REPO_ROOT / "data" / "synthetic"
PDF_OUT     = SYN_ROOT / "pdfs"
PDF_OUT.mkdir(parents=True, exist_ok=True)

MODEL       = "claude-sonnet-4-6"
MAX_TOKENS  = 4096
TEMPERATURE = 1.0
RENDER_DPI  = 200          # DPI for page → image conversion
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

api_key = os.environ.get("ANTHROPIC_API_KEY", "")
assert api_key.startswith("sk-ant"), "ANTHROPIC_API_KEY not set or invalid in .env"
CLIENT = anthropic.Anthropic(api_key=api_key)

print(f"Synthetic root : {SYN_ROOT}")
print(f"PDF output     : {PDF_OUT}")
print(f"Model          : {MODEL}")

---
## Section 1 · Synthetic Clinical Text Generation

Uses Claude to generate de-identified breast imaging + pathology reports.  
Four generation *settings* mirror the Microsoft LLM Data Creation framework:

| Setting | Seed needed | Strategy |
|---------|------------|----------|
| `naive` | Yes | Format-only seed → N new docs |
| `diverse` | No | 8 clinical profiles, cycled + shuffled |
| `similar` | Yes | Close variations of one seed |
| `tree` | Yes | Generated docs become next-round seeds |

In [ ]:
# ── Feature registry ───────────────────────────────────────────────────────────
FEATURE_DESCRIPTIONS = {
    "feature_1_lesion_size":                       "Largest lesion dimension (imaging or pathology), units preserved.",
    "feature_2_lesion_location":                   "Laterality, quadrant, clock-face, depth, distance-from-nipple.",
    "feature_3_calcifications_asymmetry":          "Presence/absence of calcifications or asymmetry; morphology and distribution.",
    "feature_4_additional_enhancement_mri":        "MRI enhancement beyond primary lesion (mass or non-mass).",
    "feature_5_extent":                            "Explicitly stated disease extent (multifocal, multicentric, bilateral, localized).",
    "feature_6_accurate_clip_placement":           "Whether a biopsy clip/marker was placed; shape and location if stated.",
    "feature_7_workup_recommendation":             "Imaging follow-up, biopsy type, surgical evaluation, or surveillance (verbatim).",
    "feature_8_lymph_node":                        "Side-specific lymph node findings or explicit no-lymphadenopathy statement.",
    "feature_9_chronology_preserved":              "Boolean: radiology studies ordered oldest → most recent.",
    "feature_10_biopsy_method":                    "Exact biopsy technique (e.g., US-guided 14g CNB, stereotactic VAB).",
    "feature_11_invasive_component_size_pathology":"Size of invasive component in cm if explicitly stated in pathology.",
    "feature_12_histologic_diagnosis":             "Exact histologic subtype(s) as written, including size/site.",
    "feature_13_receptor_status":                  "ER, PR/PgR, HER2 IHC, HER2 ISH results with category, %, intensity, controls.",
}
FEATURE_NAMES = list(FEATURE_DESCRIPTIONS.keys())

DIVERSE_PROFILES = [
    {"cancer_type": "IDC grade 2",                 "laterality": "left",  "modalities": "mammogram + ultrasound + pathology + receptor",              "special": "clip placed, calcifications present"},
    {"cancer_type": "DCIS high-grade",             "laterality": "right", "modalities": "mammogram + MRI + stereotactic biopsy + pathology",           "special": "extensive calcifications, no invasion"},
    {"cancer_type": "IDC grade 3 with DCIS",       "laterality": "left",  "modalities": "ultrasound + MRI + US-guided biopsy + pathology + receptor",  "special": "multifocal disease, axillary lymph node involvement"},
    {"cancer_type": "ILC grade 1",                 "laterality": "right", "modalities": "mammogram + MRI + US-guided biopsy + pathology",              "special": "no calcifications, non-mass MRI enhancement"},
    {"cancer_type": "IDC triple-negative grade 3", "laterality": "left",  "modalities": "ultrasound + MRI + core biopsy + pathology + receptor",       "special": "large tumor > 3 cm, no clip placed"},
    {"cancer_type": "IDC HER2-positive grade 2",   "laterality": "right", "modalities": "mammogram + ultrasound + MRI + biopsy + pathology + receptor","special": "clip placed (BB marker), low axillary node"},
    {"cancer_type": "DCIS intermediate-grade",     "laterality": "left",  "modalities": "mammogram + stereotactic biopsy + pathology",                 "special": "amorphous calcifications, localized extent"},
    {"cancer_type": "IDC grade 1 luminal A",       "laterality": "right", "modalities": "mammogram + ultrasound + US-guided biopsy + pathology + receptor", "special": "small lesion < 1 cm, no lymph node involvement"},
]

print(f"{len(FEATURE_NAMES)} features registered, {len(DIVERSE_PROFILES)} clinical profiles defined")

In [ ]:
# ── Prompt builders ────────────────────────────────────────────────────────────
_SYSTEM = textwrap.dedent("""
    You are a synthetic clinical document generator for a breast cancer AI research project.
    Write realistic, de-identified breast imaging and pathology source documents that look
    like OCR-extracted text from actual hospital reports.

    STYLE: Third person, past tense. Section headers in ALL CAPS. Fictitious dates MM/DD/YYYY.
    Use [PATIENT] for patient name. Include radiologist/pathologist sign-off lines.
    May contain occasional OCR artifacts (extra spaces, line-break mid-word).

    CLINICAL REALISM: Correct terminology, plausible sizes (0.3–5.0 cm), internally
    consistent BI-RADS, receptor status, and chronological dates.

    GROUND-TRUTH JSON: Every value must be verbatim from the document you wrote.
    Use \"Not reported\" for absent features.
""").strip()

_FEAT_LIST = "\n".join(f"  {k}: {v}" for k, v in FEATURE_DESCRIPTIONS.items())
_JSON_SCHEMA = (
    '{"lesions": [{"lesion_id": "L1", '
    + ", ".join(f'"{f}": {{"value": "...", "evidence": "..."}}'
               for f in FEATURE_NAMES if f != "feature_9_chronology_preserved")
    + '}], "feature_9_chronology_preserved": true}'
)

def _output_instructions():
    return (
        "OUTPUT FORMAT:\n"
        "[DOCUMENT TEXT — plain text, no markdown]\n\n"
        "```json\n" + _JSON_SCHEMA + "\n```"
    )

def build_diverse_prompt(profile: dict, index: int, total: int) -> str:
    return textwrap.dedent(f"""
        Generate a synthetic de-identified breast imaging + pathology source document.

        CLINICAL PROFILE (#{index} of {total}):
        - Cancer type  : {profile['cancer_type']}
        - Laterality   : {profile['laterality']}
        - Modalities   : {profile['modalities']}
        - Special notes: {profile['special']}

        Include ALL modality sections. Add realistic dates, BI-RADS, measurements,
        biopsy details, pathology findings, and receptor results.

        FEATURES TO EMBED:\n{_FEAT_LIST}

        {_output_instructions()}
    """).strip()

def build_naive_prompt(seed_text: str, index: int, total: int) -> str:
    seed = seed_text[:3000] + ("\n...[truncated]" if len(seed_text) > 3000 else "")
    return textwrap.dedent(f"""
        SEED DOCUMENT:\n{'─'*50}\n{seed}\n{'─'*50}

        Generate synthetic document #{index} of {total} with DIFFERENT clinical findings
        but the same section structure and OCR style as the seed.

        FEATURES TO EMBED:\n{_FEAT_LIST}

        {_output_instructions()}
    """).strip()

def build_similar_prompt(seed_text: str, index: int, total: int) -> str:
    seed = seed_text[:3000] + ("\n...[truncated]" if len(seed_text) > 3000 else "")
    return textwrap.dedent(f"""
        SEED DOCUMENT:\n{'─'*50}\n{seed}\n{'─'*50}

        Generate a CLOSE VARIATION (#{index} of {total}):
        - Same modality structure and section layout
        - Change measurements, receptor status, or histologic subtype plausibly
        - Preserve chronological ordering

        FEATURES TO EMBED:\n{_FEAT_LIST}

        {_output_instructions()}
    """).strip()

print("Prompt builders defined")

In [ ]:
# ── Core generation utilities ──────────────────────────────────────────────────
def call_claude(prompt: str) -> str:
    msg = CLIENT.messages.create(
        model=MODEL, max_tokens=MAX_TOKENS, temperature=TEMPERATURE,
        system=_SYSTEM,
        messages=[{"role": "user", "content": prompt}],
    )
    return msg.content[0].text

def parse_output(raw: str) -> tuple:
    """Split Claude response into (doc_text, features_dict)."""
    m = re.search(r"```json\s*(\{.*?\})\s*```", raw, re.DOTALL)
    if m:
        doc_text = raw[:m.start()].strip()
        try:
            features = json.loads(m.group(1))
        except json.JSONDecodeError:
            features = None
    else:
        doc_text, features = raw.strip(), None
    return doc_text, features

def done_indices(out_dir: Path, setting: str) -> set:
    """Return already-saved doc indices for resume support."""
    prefix = setting.split("_")[0]
    return {
        int(m.group(1))
        for p in out_dir.glob(f"{prefix}_*_document.txt")
        if (m := re.match(rf"{re.escape(prefix)}_(\d{{4}})_", p.name))
    }

def save_result(doc_text: str, features, out_dir: Path,
                setting: str, index: int, meta: dict = None) -> Path:
    doc_id   = f"{setting}_{index:04d}_{uuid.uuid4().hex[:6]}"
    txt_path = out_dir / f"{doc_id}_document.txt"
    txt_path.write_text(doc_text, encoding="utf-8")
    payload = {"doc_id": doc_id, "setting": setting, "index": index,
               "profile": meta or {}, "features": features,
               "json_parse_ok": features is not None}
    (out_dir / f"{doc_id}_features.json").write_text(
        json.dumps(payload, indent=2), encoding="utf-8")
    return txt_path

def load_seeds(source_dir: Path, max_seeds: int = 50) -> list:
    files = sorted(source_dir.glob("*.txt"))[:max_seeds]
    return [f.read_text(encoding="utf-8", errors="replace") for f in files]

print("Utilities defined")

In [ ]:
# ── Run text generation ────────────────────────────────────────────────────────
# Adjust SETTING and N_DOCS to your needs.
# 'diverse' requires no seeds; 'naive' / 'similar' use OCR_CACHE as seeds.

SETTING  = "diverse"   # naive | diverse | similar
N_DOCS   = 10          # number to generate this run

out_dir  = SYN_ROOT / SETTING
out_dir.mkdir(parents=True, exist_ok=True)

seeds    = load_seeds(OCR_CACHE) if SETTING in ("naive", "similar") else []
done     = done_indices(out_dir, SETTING)
profiles = (DIVERSE_PROFILES * (N_DOCS // len(DIVERSE_PROFILES) + 1))
if SETTING == "diverse":
    random.shuffle(profiles)

generated_paths = []

for i in range(1, N_DOCS + 1):
    if i in done:
        print(f"  [{SETTING}] skipping {i}/{N_DOCS} (already saved)")
        continue

    if SETTING == "diverse":
        profile = profiles[i - 1]
        prompt  = build_diverse_prompt(profile, i, N_DOCS)
        label   = profile["cancer_type"]
    elif SETTING == "naive":
        profile, label = {}, ""
        prompt  = build_naive_prompt(random.choice(seeds), i, N_DOCS)
    else:  # similar
        profile, label = {}, ""
        prompt  = build_similar_prompt(random.choice(seeds), i, N_DOCS)

    print(f"  [{SETTING}] {i}/{N_DOCS} {label} ...", end=" ", flush=True)
    raw  = call_claude(prompt)
    doc, feats = parse_output(raw)
    path = save_result(doc, feats, out_dir, SETTING, i, profile)
    generated_paths.append(path)
    print(f"saved  (json_ok={feats is not None})")

print(f"\nTotal new docs saved: {len(generated_paths)}")
json_ok = sum(1 for p in out_dir.glob("*_features.json")
              if json.loads(p.read_text()).get("json_parse_ok"))
total_f = sum(1 for _ in out_dir.glob("*_features.json"))
print(f"JSON parse success : {json_ok}/{total_f} across all docs in {SETTING}/")

In [ ]:
# ── Preview one generated document ────────────────────────────────────────────
sample = sorted(out_dir.glob("*_document.txt"))[0]
text   = sample.read_text(encoding="utf-8")
print(f"File : {sample.name}  ({len(text):,} chars)")
print("─" * 70)
print(text[:2500])

---
## Section 2 · Synthetic Scanned PDF Generation

Each synthetic `.txt` file is converted to a realistic-looking scanned PDF in three steps:

1. **`text_to_clean_pdf`** — `reportlab` renders clinical text with Courier font, proper margins, and section-header bold formatting.
2. **`apply_scan_degradation`** — `cv2` applies noise, blur, skew, and contrast reduction at three quality levels.
3. **`clean_pdf_to_scanned_pdf`** — pages are rendered to images via `fitz`, degraded, and re-assembled into a PDF.

| Quality | Noise σ | Blur kernel | Skew | Contrast |
|---------|---------|-------------|------|----------|
| `good`  | 5       | none        | ±0.5° | 1.00   |
| `medium`| 15      | 3×3         | ±1.5° | 0.90   |
| `poor`  | 30      | 5×5         | ±3.5° | 0.75   |

In [ ]:
# ── Step 1: text → clean PDF (reportlab) ──────────────────────────────────────
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_LEFT, TA_CENTER
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, HRFlowable
from reportlab.lib.pagesizes import LETTER
from reportlab.lib.units import inch

def text_to_clean_pdf(text: str, out_path: Path) -> Path:
    """Render OCR-style clinical text to a clean monospaced PDF."""
    doc = SimpleDocTemplate(
        str(out_path),
        pagesize=LETTER,
        leftMargin=0.9*inch, rightMargin=0.9*inch,
        topMargin=1.0*inch,  bottomMargin=1.0*inch,
    )

    # Styles
    normal = ParagraphStyle(
        "Normal", fontName="Courier", fontSize=9, leading=13, spaceAfter=2,
        alignment=TA_LEFT,
    )
    header = ParagraphStyle(
        "Header", fontName="Courier-Bold", fontSize=9, leading=13, spaceAfter=2,
        alignment=TA_LEFT,
    )

    def is_section_header(line: str) -> bool:
        stripped = line.strip()
        return bool(stripped and stripped == stripped.upper()
                    and len(stripped) > 3 and not stripped[0].isdigit())

    story = []
    for line in text.splitlines():
        safe = line.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
        if not safe.strip():
            story.append(Spacer(1, 4))
        elif is_section_header(safe):
            story.append(Spacer(1, 6))
            story.append(Paragraph(safe, header))
        else:
            story.append(Paragraph(safe, normal))

    doc.build(story)
    return out_path

# Quick smoke test
sample_txt = sorted(out_dir.glob("*_document.txt"))[0]
_tmp = PDF_OUT / "_test_clean.pdf"
text_to_clean_pdf(sample_txt.read_text(encoding="utf-8"), _tmp)
print(f"Clean PDF written: {_tmp}  ({_tmp.stat().st_size:,} bytes)")

In [ ]:
# ── Step 2: scan degradation pipeline ─────────────────────────────────────────
SCAN_PROFILES = {
    "good":   dict(noise_std=5,  blur_k=0, skew_deg=0.5, contrast=1.00, brightness_cap=255),
    "medium": dict(noise_std=15, blur_k=3, skew_deg=1.5, contrast=0.90, brightness_cap=245),
    "poor":   dict(noise_std=30, blur_k=5, skew_deg=3.5, contrast=0.75, brightness_cap=230),
}

def apply_scan_degradation(img_gray: np.ndarray, quality: str = "medium",
                           rng: np.random.Generator = None) -> np.ndarray:
    """Apply realistic scanner artifacts to a grayscale page image."""
    if rng is None:
        rng = np.random.default_rng()
    cfg = SCAN_PROFILES[quality]
    img = img_gray.astype(np.float32)

    # 1. Gaussian noise
    if cfg["noise_std"] > 0:
        img += rng.normal(0, cfg["noise_std"], img.shape)

    # 2. Contrast + brightness reduction (simulate faded paper)
    img = img * cfg["contrast"]
    img = np.clip(img, 0, cfg["brightness_cap"]).astype(np.uint8)

    # 3. Gaussian blur (simulate focus or paper texture)
    if cfg["blur_k"] > 0:
        k = cfg["blur_k"] | 1   # ensure odd
        img = cv2.GaussianBlur(img, (k, k), 0)

    # 4. Random skew (simulate misaligned paper feed)
    if cfg["skew_deg"] > 0:
        h, w = img.shape
        angle = float(rng.uniform(-cfg["skew_deg"], cfg["skew_deg"]))
        M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
        img = cv2.warpAffine(img, M, (w, h),
                             flags=cv2.INTER_LINEAR, borderValue=255)

    # 5. Subtle paper yellowing (add warm tint when converting to RGB later)
    img._yellow_tint = quality  # marker for downstream RGB conversion
    return img


def gray_to_yellowed_rgb(img_gray: np.ndarray, quality: str) -> np.ndarray:
    """Convert degraded grayscale to slightly yellowed RGB to mimic aged paper."""
    tints = {"good": (0, 0, 0), "medium": (5, 5, -8), "poor": (12, 10, -20)}
    rgb = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2BGR).astype(np.int16)
    dr, dg, db = tints[quality]   # BGR offsets
    rgb[:, :, 0] = np.clip(rgb[:, :, 0] + db, 0, 255)
    rgb[:, :, 1] = np.clip(rgb[:, :, 1] + dg, 0, 255)
    rgb[:, :, 2] = np.clip(rgb[:, :, 2] + dr, 0, 255)
    return rgb.astype(np.uint8)

print("Scan degradation pipeline defined")

In [ ]:
# ── Step 3: clean PDF → scanned PDF ───────────────────────────────────────────
def clean_pdf_to_scanned_pdf(clean_pdf: Path, out_path: Path,
                              quality: str = "medium", dpi: int = RENDER_DPI) -> Path:
    """Render clean PDF pages to images, degrade them, reassemble as PDF."""
    rng = np.random.default_rng(seed=hash(clean_pdf.name) & 0xFFFFFFFF)
    doc = fitz.open(str(clean_pdf))
    pil_pages = []

    for page in doc:
        zoom = dpi / 72.0
        mat  = fitz.Matrix(zoom, zoom)
        pix  = page.get_pixmap(matrix=mat, alpha=False, colorspace=fitz.csGRAY)
        arr  = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width)
        arr  = apply_scan_degradation(arr, quality=quality, rng=rng)
        rgb  = gray_to_yellowed_rgb(arr, quality)
        pil_pages.append(Image.fromarray(rgb))

    doc.close()

    if pil_pages:
        pil_pages[0].save(
            str(out_path), save_all=True,
            append_images=pil_pages[1:], resolution=dpi,
        )
    return out_path


# Smoke test — generate all three quality levels for the sample doc
sample_txt = sorted(out_dir.glob("*_document.txt"))[0]
clean_pdf  = PDF_OUT / f"_test_clean.pdf"
text_to_clean_pdf(sample_txt.read_text(encoding="utf-8"), clean_pdf)

for q in ("good", "medium", "poor"):
    out = PDF_OUT / f"_test_{q}.pdf"
    clean_pdf_to_scanned_pdf(clean_pdf, out, quality=q)
    print(f"  [{q}] {out.name}  {out.stat().st_size:,} bytes")

print("\nSmoke test passed — check data/synthetic/pdfs/ for output files.")

In [ ]:
# ── Visualise: side-by-side clean vs degraded pages ───────────────────────────
def show_page_comparison(clean_pdf: Path, quality: str = "medium",
                         page_index: int = 0, dpi: int = 100):
    zoom = dpi / 72.0
    mat  = fitz.Matrix(zoom, zoom)
    doc  = fitz.open(str(clean_pdf))
    pix  = doc[page_index].get_pixmap(matrix=mat, alpha=False, colorspace=fitz.csGRAY)
    doc.close()
    clean_arr  = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width)
    rng        = np.random.default_rng(42)
    degrad_arr = apply_scan_degradation(clean_arr.copy(), quality=quality, rng=rng)
    degrad_rgb = gray_to_yellowed_rgb(degrad_arr, quality)

    fig, axes = plt.subplots(1, 2, figsize=(14, 9))
    axes[0].imshow(clean_arr,  cmap="gray", vmin=0, vmax=255)
    axes[0].set_title("Clean PDF render",        fontsize=13)
    axes[0].axis("off")
    axes[1].imshow(degrad_rgb[:, :, ::-1])  # BGR → RGB for matplotlib
    axes[1].set_title(f"Scanned ({quality} quality)", fontsize=13)
    axes[1].axis("off")
    plt.suptitle(clean_pdf.stem, fontsize=10, y=0.02)
    plt.tight_layout()
    plt.show()

show_page_comparison(clean_pdf, quality="medium")

In [ ]:
# ── Show all three quality levels ─────────────────────────────────────────────
doc   = fitz.open(str(clean_pdf))
zoom  = 100 / 72.0
mat   = fitz.Matrix(zoom, zoom)
pix   = doc[0].get_pixmap(matrix=mat, alpha=False, colorspace=fitz.csGRAY)
doc.close()
clean_arr = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width)

fig, axes = plt.subplots(1, 4, figsize=(20, 8))
axes[0].imshow(clean_arr, cmap="gray");  axes[0].set_title("Clean"); axes[0].axis("off")
for ax, q in zip(axes[1:], ("good", "medium", "poor")):
    rng = np.random.default_rng(42)
    arr = apply_scan_degradation(clean_arr.copy(), quality=q, rng=rng)
    rgb = gray_to_yellowed_rgb(arr, q)
    ax.imshow(rgb[:, :, ::-1])
    ax.set_title(f"Scan — {q}")
    ax.axis("off")
plt.suptitle("Scan quality comparison", fontsize=13)
plt.tight_layout()
plt.show()

---
## Section 3 · Batch PDF Generation

Process all synthetic `.txt` files in the chosen setting directory into scanned PDFs.

In [ ]:
# ── Batch: txt → scanned PDF at all quality levels ────────────────────────────
QUALITIES     = ["good", "medium", "poor"]
BATCH_SETTING = SETTING   # change to 'naive', 'similar', etc. if needed

txt_files = sorted((SYN_ROOT / BATCH_SETTING).glob("*_document.txt"))
print(f"Found {len(txt_files)} synthetic text files in '{BATCH_SETTING}/'")

summary = []
for txt_path in txt_files:
    doc_id    = txt_path.stem.replace("_document", "")
    clean_pdf = PDF_OUT / f"{doc_id}_clean.pdf"

    # Render clean PDF (skip if already done)
    if not clean_pdf.exists():
        text_to_clean_pdf(txt_path.read_text(encoding="utf-8"), clean_pdf)

    row = {"doc_id": doc_id}
    for q in QUALITIES:
        scan_pdf = PDF_OUT / f"{doc_id}_{q}.pdf"
        if not scan_pdf.exists():
            clean_pdf_to_scanned_pdf(clean_pdf, scan_pdf, quality=q)
        row[f"{q}_kb"] = round(scan_pdf.stat().st_size / 1024, 1)

    summary.append(row)
    print(f"  {doc_id}  good={row['good_kb']}KB  medium={row['medium_kb']}KB  poor={row['poor_kb']}KB")

print(f"\nDone. {len(summary)} docs × {len(QUALITIES)} qualities = {len(summary)*len(QUALITIES)} PDFs")

---
## Section 4 · OCR Quality Verification

Run the project's existing OCR quality metrics (Laplacian variance / sharpness, RMS contrast, skew angle) on the generated PDFs to confirm the degradation pipeline produces the expected quality spread.

In [ ]:
# ── OCR quality metrics (adapted from ocr_quality_scoring.py) ─────────────────
def laplacian_variance(gray: np.ndarray) -> float:
    """Sharpness proxy — higher = sharper."""
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())

def rms_contrast(gray: np.ndarray) -> float:
    return float(gray.astype(np.float32).std())

def tenengrad(gray: np.ndarray) -> float:
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    return float(np.mean(gx**2 + gy**2))

def score_pdf_page(pdf_path: Path, page_idx: int = 0, dpi: int = 150) -> dict:
    doc  = fitz.open(str(pdf_path))
    zoom = dpi / 72.0
    pix  = doc[page_idx].get_pixmap(matrix=fitz.Matrix(zoom, zoom),
                                    alpha=False, colorspace=fitz.csGRAY)
    doc.close()
    gray = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width)
    return {
        "laplacian_var": round(laplacian_variance(gray), 2),
        "rms_contrast":  round(rms_contrast(gray), 2),
        "tenengrad":     round(tenengrad(gray), 2),
        "mean_bright":   round(float(gray.mean()), 2),
    }

print("Quality metric functions defined")

In [ ]:
# ── Score all generated PDFs ───────────────────────────────────────────────────
import pandas as pd

records = []
for row in summary:
    doc_id = row["doc_id"]
    for q in QUALITIES:
        pdf_path = PDF_OUT / f"{doc_id}_{q}.pdf"
        if pdf_path.exists():
            metrics = score_pdf_page(pdf_path)
            records.append({"doc_id": doc_id, "quality": q, **metrics})

df_qual = pd.DataFrame(records)
print(df_qual.groupby("quality")[["laplacian_var", "rms_contrast", "tenengrad", "mean_bright"]]
             .mean().round(2).to_string())

In [ ]:
# ── Visualise OCR metric distributions ────────────────────────────────────────
import seaborn as sns
sns.set_style("whitegrid")

metrics_to_plot = ["laplacian_var", "rms_contrast", "tenengrad"]
titles = ["Laplacian Variance (Sharpness)", "RMS Contrast", "Tenengrad (Edge Energy)"]
order  = ["good", "medium", "poor"]
palette = {"good": "#2ecc71", "medium": "#f39c12", "poor": "#e74c3c"}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, metric, title in zip(axes, metrics_to_plot, titles):
    sns.boxplot(data=df_qual, x="quality", y=metric, order=order,
                palette=palette, ax=ax, width=0.5)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Scan Quality")
    ax.set_ylabel(metric)

plt.suptitle("OCR Quality Metrics by Scan Level — Synthetic PDFs", fontsize=13, y=1.02)
plt.tight_layout()

plot_path = REPO_ROOT / "reports" / "synthetic_pdf_ocr_quality.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved: {plot_path}")

---
## Summary

| Output | Location |
|--------|----------|
| Synthetic `.txt` + feature JSON | `data/synthetic/{setting}/` |
| Clean PDFs | `data/synthetic/pdfs/{id}_clean.pdf` |
| Scanned PDFs (3 quality levels) | `data/synthetic/pdfs/{id}_{good,medium,poor}.pdf` |
| OCR quality distribution plot | `reports/synthetic_pdf_ocr_quality.png` |

**Next steps:**
- Run the OCR preprocessing pipeline (`08_ocr_image_quality_deblur.ipynb`) on the `poor` quality PDFs to validate recovery.
- Use the ground-truth feature JSON to evaluate extraction accuracy on synthetic inputs.
- Scale up generation by increasing `N_DOCS` (resume-safe — already-saved docs are skipped automatically).